In [11]:
"""
============================================================
 Скрипт выгрузки 10 датасетов для прогнозирования
============================================================
 Группа A: Нефть WTI, Природный газ, Золото
 Группа B: EUR/USD, GBP/USD, USD/JPY
 Группа C: CPI, Industrial Production, UMCSENT
 Группа D: Bitcoin, Ethereum
 + предикторы для каждой группы
============================================================
 Источники:
   - yfinance         : рыночные цены, крипто
   - pandas_datareader: FRED (макро США), EIA
   - fredapi          : FRED прямой API (запасной)
============================================================
"""

import os
import warnings
import time
import logging
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

# ─── настройка логирования ────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)-7s │ %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ─── параметры ────────────────────────────────────────────────────────────────
START_DATE = "2010-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")
OUTPUT_DIR = Path("datasets")
OUTPUT_DIR.mkdir(exist_ok=True)

FRED_API_KEY = os.getenv("FRED_API_KEY", "cf3eada423161498d2d12bf0ec2a2a84")   
EIA_API_KEY = os.getenv("EIA_API_KEY", "rJd8r6aprfbUkV7yuQBC9eJaP6RGo4M5SzqUjawO")   


# Вспомогательные функции

In [9]:
def fetch_eia(series: dict, start: str, end: str) -> pd.DataFrame:
    """
    Скачивает еженедельные petroleum серии из EIA API v2.
    Требует бесплатный ключ с https://www.eia.gov/opendata/

    Ключевые series_id для группы A:
      W_EPC0_SAX_YCUOK_MBBL  — запасы нефти в Кушинге (тыс. барр.)
      WCRFPUS2               — добыча нефти США (тыс. барр./сут)
      WPULEUS3               — загрузка НПЗ США (%)
      WCESTUS1               — суммарные запасы нефти США (тыс. барр.)
      NGSCSUS1               — запасы газа США (млрд куб. фут)
    """
    if not EIA_API_KEY:
        log.warning("  EIA_API_KEY не задан — EIA серии пропущены. "
                    "Бесплатный ключ: https://www.eia.gov/opendata/")
        return pd.DataFrame()

    import urllib.request as _ur
    import json as _json

    # Роутер по endpoint-у EIA API v2
    STOC_WSTK = {"W_EPC0_SAX_YCUOK_MBBL", "WCESTUS1"}
    GAS_WKLY  = {"NGSCSUS1"}

    dfs = {}
    for name, sid in series.items():
        log.info(f"  EIA API: {sid} → {name}")
        fetched = False

        # Попытка 1: EIA API v2 (route по типу серии)
        if sid in STOC_WSTK:
            v2_url = (
                "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"
                f"?api_key={EIA_API_KEY}&frequency=weekly&data[0]=value"
                f"&facets[series][]={sid}&start={start[:7]}&length=2000"
                "&sort[0][column]=period&sort[0][direction]=asc"
            )
        elif sid in GAS_WKLY:
            v2_url = (
                "https://api.eia.gov/v2/natural-gas/stor/wkly/data/"
                f"?api_key={EIA_API_KEY}&frequency=weekly&data[0]=value"
                f"&facets[series][]={sid}&start={start[:7]}&length=2000"
                "&sort[0][column]=period&sort[0][direction]=asc"
            )
        else:
            v2_url = None  # производство / загрузка НПЗ — через v1

        # Попытка 2: EIA API v1 (универсальный, все PET серии)
        v1_url = (
            f"https://api.eia.gov/series/?api_key={EIA_API_KEY}"
            f"&series_id=PET.{sid}.W"
        )

        for url in filter(None, [v2_url, v1_url]):
            try:
                req = _ur.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                with _ur.urlopen(req, timeout=20) as r:
                    data = _json.loads(r.read())

                # v2 формат
                if "response" in data:
                    rows = data["response"].get("data", [])
                    if rows:
                        df = pd.DataFrame(rows)
                        df.index = pd.to_datetime(df["period"])
                        df = df[["value"]].rename(columns={"value": name})
                        df[name] = pd.to_numeric(df[name], errors="coerce")
                        df = df[df.index >= start].sort_index()
                        dfs[name] = df
                        fetched = True
                        break

                # v1 формат
                elif "series" in data and data["series"]:
                    rows = data["series"][0]["data"]   # [[datestr, value], ...]
                    df = pd.DataFrame(rows, columns=["date", name])
                    df.index = pd.to_datetime(df["date"])
                    df = df[[name]]
                    df[name] = pd.to_numeric(df[name], errors="coerce")
                    df = df[df.index >= start].sort_index()
                    dfs[name] = df
                    fetched = True
                    break

            except Exception as e:
                log.warning(f"    EIA {url[:70]}... ошибка: {e}")

        if not fetched:
            log.error(f"    Не удалось загрузить EIA серию {sid}")
        time.sleep(0.4)

    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs.values(), axis=1)


def fetch_eia_with_fallback(
    eia_series: dict,
    fred_fallback: dict,
    start: str,
    end: str,
) -> pd.DataFrame:
    """
    Логика fallback: EIA weekly (приоритет) → FRED monthly → пропуск.

    eia_series:    {name: eia_series_id}
    fred_fallback: {name: fred_series_id | None}
      None означает «нет аналога в FRED, пропустить»
    """
    # Шаг 1: загрузить EIA
    eia_df = fetch_eia(eia_series, start, end)
    eia_cols = set(eia_df.columns) if not eia_df.empty else set()

    # Шаг 2: для недостающих серий — FRED fallback
    fallback_map = {
        name: fred_id
        for name, fred_id in fred_fallback.items()
        if name not in eia_cols and fred_id is not None
    }
    fred_df = pd.DataFrame()
    if fallback_map:
        log.info(f"  FRED fallback для: {list(fallback_map.keys())}")
        fred_df = fetch_fred(fallback_map, start, end)

    # Шаг 3: объединить результаты
    parts = [df for df in [eia_df, fred_df] if not df.empty]
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, axis=1)

In [6]:
def fetch_yf(tickers: dict, start: str, end: str, price_col: str = "Close") -> pd.DataFrame:
    """Скачивает цены закрытия для списка тикеров через yfinance."""
    dfs = {}
    for name, ticker in tickers.items():
        log.info(f"  yfinance: {ticker} → {name}")
        try:
            raw = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if raw.empty:
                log.warning(f"    Нет данных для {ticker}")
                continue
            # yfinance может вернуть MultiIndex columns
            if isinstance(raw.columns, pd.MultiIndex):
                raw = raw[price_col][ticker]
            else:
                raw = raw[price_col]
            raw.name = name
            dfs[name] = raw
        except Exception as e:
            log.error(f"    Ошибка {ticker}: {e}")
        time.sleep(0.3)  # вежливая пауза
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs.values(), axis=1)


def fetch_fred(series: dict, start: str, end: str) -> pd.DataFrame:
    """
    Скачивает серии из FRED.
    Приоритет: fredapi (если задан FRED_API_KEY) → pandas_datareader → requests напрямую.
    """
    if FRED_API_KEY:
        return fetch_fred_api(series, start, end)
    return fetch_fred_requests(series, start, end)


def fetch_fred_requests(series: dict, start: str, end: str) -> pd.DataFrame:
    """
    Прямой HTTP-запрос к FRED API без внешних библиотек.
    Работает без API-ключа (анонимно) через публичный endpoint observations.
    """
    import urllib.request, json
    BASE = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="
    dfs = {}
    for name, code in series.items():
        log.info(f"  FRED (direct): {code} → {name}")
        try:
            url = f"{BASE}{code}&vintage_date={end}"
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=15) as r:
                from io import StringIO
                text = r.read().decode("utf-8")
            raw = pd.read_csv(StringIO(text), index_col=0, parse_dates=True)
            raw.columns = [name]
            raw = raw[raw.index >= start]
            raw.replace(".", np.nan, inplace=True)
            raw[name] = pd.to_numeric(raw[name], errors="coerce")
            dfs[name] = raw
        except Exception as e:
            log.error(f"    Ошибка FRED direct {code}: {e}")
        time.sleep(0.4)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs.values(), axis=1)


def fetch_fred_api(series: dict, start: str, end: str) -> pd.DataFrame:
    """Запасной метод — fredapi (требует API ключ)."""
    if not FRED_API_KEY:
        log.error("FRED_API_KEY не задан. Установите через: export FRED_API_KEY=your_key")
        return pd.DataFrame()
    try:
        from fredapi import Fred
        fred = Fred(api_key=FRED_API_KEY)
        dfs = {}
        for name, code in series.items():
            log.info(f"  fredapi: {code} → {name}")
            try:
                raw = fred.get_series(code, observation_start=start, observation_end=end)
                raw.name = name
                dfs[name] = raw
            except Exception as e:
                log.error(f"    Ошибка fredapi {code}: {e}")
        if not dfs:
            return pd.DataFrame()
        return pd.concat(dfs.values(), axis=1)
    except ImportError:
        log.error("fredapi не установлен: pip install fredapi")
        return pd.DataFrame()


def resample_to_freq(df: pd.DataFrame, freq: str = "D",
                     method: str = "ffill") -> pd.DataFrame:
    """Приводит датафрейм к единой частоте (D/W/ME)."""
    if df.empty:
        return df
    df = df.sort_index()
    df.index = pd.to_datetime(df.index)
    if freq == "D":
        idx = pd.date_range(df.index.min(), df.index.max(), freq="B")  # рабочие дни
    elif freq == "W":
        idx = pd.date_range(df.index.min(), df.index.max(), freq="W-FRI")
    else:
        idx = pd.date_range(df.index.min(), df.index.max(), freq="ME")
    return df.reindex(idx).ffill().bfill()


def add_technical_features(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Добавляет стандартный набор технических предикторов для ценового ряда:
    лаги, скользящие средние, волатильность, RSI, MACD.
    """
    if col not in df.columns:
        return df

    s = df[col].copy()

    # Лаги
    for lag in [1, 2, 3, 5, 10, 21]:
        df[f"{col}_lag{lag}"] = s.shift(lag)

    # Доходности
    df[f"{col}_ret1"]  = s.pct_change(1)
    df[f"{col}_ret5"]  = s.pct_change(5)
    df[f"{col}_ret21"] = s.pct_change(21)

    # Скользящие средние и Z-score
    for w in [5, 10, 21, 63]:
        df[f"{col}_ma{w}"]    = s.rolling(w).mean()
        df[f"{col}_std{w}"]   = s.rolling(w).std()
        df[f"{col}_zscore{w}"] = (s - df[f"{col}_ma{w}"]) / (df[f"{col}_std{w}"] + 1e-9)

    # RSI (14)
    delta = s.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / (loss + 1e-9)
    df[f"{col}_rsi14"] = 100 - 100 / (1 + rs)

    # MACD
    ema12 = s.ewm(span=12, adjust=False).mean()
    ema26 = s.ewm(span=26, adjust=False).mean()
    df[f"{col}_macd"]        = ema12 - ema26
    df[f"{col}_macd_signal"] = df[f"{col}_macd"].ewm(span=9, adjust=False).mean()
    df[f"{col}_macd_hist"]   = df[f"{col}_macd"] - df[f"{col}_macd_signal"]

    # Bollinger Bands (20)
    ma20  = s.rolling(20).mean()
    std20 = s.rolling(20).std()
    df[f"{col}_bb_upper"] = ma20 + 2 * std20
    df[f"{col}_bb_lower"] = ma20 - 2 * std20
    df[f"{col}_bb_pct"]   = (s - df[f"{col}_bb_lower"]) / (
        df[f"{col}_bb_upper"] - df[f"{col}_bb_lower"] + 1e-9)

    return df


def save_dataset(df: pd.DataFrame, name: str, group: str) -> None:
    """Сохраняет датасет в CSV."""
    if df.empty:
        log.warning(f"  Датасет {name} пустой, пропускаем")
        return
    path = OUTPUT_DIR / f"{group}_{name}.csv"
    df.to_csv(path)
    rows, cols = df.shape
    log.info(f"  ✓ Сохранён: {path.name}  [{rows} строк × {cols} столбцов]")

# ГРУППА A: ЭНЕРГЕТИКА И МЕТАЛЛЫ

In [ ]:
def build_group_a():
    """
    Целевые ряды:
      - Нефть WTI (CL=F)
      - Природный газ Henry Hub (NG=F)
      - Золото (GC=F)

    Предикторы:
      - Нефть Brent (BZ=F)                     — спред WTI/Brent
      - Доллар DXY (DX-Y.NYB)                  — валютный эффект
      - S&P 500 (^GSPC)                         — risk-on/off
      - VIX (^VIX)                              — волатильность рынка
      - Медь (HG=F)                             — промышленный спрос
      - Treasury 10Y yield (^TNX)               — ставки / discount rate
      - EIA  (weekly, приоритет):
          Cushing stocks   W_EPC0_SAX_YCUOK_MBBL  ← центральный индикатор WTI
          добыча нефти     WCRFPUS2               ← предложение
          загрузка НПЗ     WPULEUS3               ← прямой спрос
          запасы нефти США WCESTUS1
          запасы газа США  NGSCSUS1
      - FRED (monthly, fallback при отсутствии EIA ключа):
          ISM PMI          ISAPMFG
          IP добыча нефти  IPG21112S
          загрузка НПЗ     CAPUTLG32411S
      - Технические фичи по каждому целевому ряду
      EIA ключ (бесплатно): https://www.eia.gov/opendata/
    """
    log.info("\n═══ ГРУППА A: Энергетика и Металлы ═══")

    # ── целевые ряды ──────────────────────────────────────────────────────────
    targets = fetch_yf(
        {"WTI_oil": "CL=F", "NatGas": "NG=F", "Gold": "GC=F"},
        START_DATE, END_DATE
    )

    # ── рыночные предикторы ───────────────────────────────────────────────────
    market_pred = fetch_yf(
        {
            "Brent":     "BZ=F",
            "DXY":       "DX-Y.NYB",
            "SP500":     "^GSPC",
            "VIX":       "^VIX",
            "Copper":    "HG=F",
            "Yield10Y":  "^TNX",
            "Silver":    "SI=F",
        },
        START_DATE, END_DATE
    )

    # ── макро FRED (месячные; работают через fredapi без EIA ключа) ───────────
    fred_pred = fetch_fred(
        {
            "IndProd_USA":  "INDPRO",      # промпроизводство США (monthly)
            "InflationCPI": "CPIAUCSL",    # CPI (monthly)
            #"ISM_Mfg":     "ISAPMFG",     # ISM Manufacturing Index (NAPM устарел)
        },
        START_DATE, END_DATE
    )

    # ── EIA weekly → FRED monthly fallback ────────────────────────────────────
    eia_pred = fetch_eia_with_fallback(
        eia_series={
            "Cushing_stocks":  "W_EPC0_SAX_YCUOK_MBBL",  # запасы в Кушинге (тыс. барр.)
            #"US_oil_prod_w":   "WCRFPUS2",                # добыча нефти США (тыс. барр./сут)
            #"Refinery_util_w": "WPULEUS3",                # загрузка НПЗ США (%)
            "EIA_crude_total": "WCESTUS1",                # суммарные запасы нефти США
            "EIA_gas_stocks":  "NGSCSUS1",                # запасы газа США
        },
        fred_fallback={
            "Cushing_stocks":  None,             # нет прямого аналога в FRED
            "US_oil_prod_w":   "IPG21112S",      # IP: добыча нефти — индекс (SA)
            "Refinery_util_w": "CAPUTLG32411S",  # Capacity Utilization: НПЗ (SA)
            "EIA_crude_total": None,             # нет в FRED
            "EIA_gas_stocks":  None,             # нет в FRED
        },
        start=START_DATE, end=END_DATE,
    )

    # ── объединяем на дневной сетке ───────────────────────────────────────────
    base = pd.concat([targets, market_pred], axis=1).sort_index()
    if not fred_pred.empty:
        fred_pred = resample_to_freq(fred_pred, "D")
        base = base.join(fred_pred, how="left").ffill()
    if not eia_pred.empty:
        eia_pred = resample_to_freq(eia_pred, "D")
        base = base.join(eia_pred, how="left").ffill()

    # ── технические фичи ──────────────────────────────────────────────────────
    #for col in ["WTI_oil", "NatGas", "Gold"]:
    #    base = add_technical_features(base, col)

    # Спред WTI–Brent
    if "WTI_oil" in base.columns and "Brent" in base.columns:
        base["WTI_Brent_spread"] = base["WTI_oil"] - base["Brent"]

    # ── сохраняем 3 датасета ──────────────────────────────────────────────────
    common_pred = [c for c in base.columns
                   if not c.startswith(("WTI_", "NatGas", "Gold"))]

    for target, prefix in [("WTI_oil", "WTI_"), ("NatGas", "NatGas"), ("Gold", "Gold")]:
        if target not in base.columns:
            continue
        target_cols = [c for c in base.columns
                       if c == target or c.startswith(prefix)]
        ds = base[target_cols + common_pred].dropna(subset=[target])
        save_dataset(ds, target, "A")


In [18]:
build_group_a()

14:01:13 │ INFO    │ 
═══ ГРУППА A: Энергетика и Металлы ═══
14:01:13 │ INFO    │   yfinance: CL=F → WTI_oil
14:01:13 │ INFO    │   yfinance: NG=F → NatGas
14:01:14 │ INFO    │   yfinance: GC=F → Gold
14:01:14 │ INFO    │   yfinance: BZ=F → Brent
14:01:14 │ INFO    │   yfinance: DX-Y.NYB → DXY
14:01:15 │ INFO    │   yfinance: ^GSPC → SP500
14:01:15 │ INFO    │   yfinance: ^VIX → VIX
14:01:15 │ INFO    │   yfinance: HG=F → Copper
14:01:16 │ INFO    │   yfinance: ^TNX → Yield10Y
14:01:16 │ INFO    │   yfinance: SI=F → Silver
14:01:17 │ INFO    │   fredapi: INDPRO → IndProd_USA
14:01:17 │ INFO    │   fredapi: CPIAUCSL → InflationCPI
14:01:17 │ INFO    │   EIA API: W_EPC0_SAX_YCUOK_MBBL → Cushing_stocks
14:01:28 │ INFO    │   EIA API: WCRFPUS2 → US_oil_prod_w
14:01:32 │ WARNING │     EIA https://api.eia.gov/series/?api_key=rJd8r6aprfbUkV7yuQBC9eJaP6RGo4M5Sz... ошибка: HTTP Error 404: Not Found
14:01:32 │ ERROR   │     Не удалось загрузить EIA серию WCRFPUS2
14:01:33 │ INFO    │   EIA API: 

# ГРУППА B: ВАЛЮТНЫЙ РЫНОК

In [ ]:
def build_group_b():
    """
    Целевые ряды:
      - EUR/USD (EURUSD=X)
      - GBP/USD (GBPUSD=X)
      - USD/JPY (USDJPY=X)

    Предикторы:
      - DXY (индекс доллара)
      - Другие валютные пары (кросс-эффекты)
      - VIX (риск-аппетит)
      - Gold (safe haven)
      - Treasury 10Y vs Bund spread (^TNX, реплика через DE10Y)
      - S&P 500, Nikkei
      - FRED: дифференциал ставок (Fed Funds vs ECB proxy через M2SL, FEDFUNDS)
      - Технические фичи
    """
    log.info("\n═══ ГРУППА B: Валютный рынок ═══")

    targets = fetch_yf(
        {"EURUSD": "EURUSD=X", "GBPUSD": "GBPUSD=X", "USDJPY": "USDJPY=X"},
        START_DATE, END_DATE
    )

    market_pred = fetch_yf(
        {
            "DXY":       "DX-Y.NYB",
            "VIX":       "^VIX",
            "Gold":      "GC=F",
            "WTI":       "CL=F",
            "SP500":     "^GSPC",
            "Nikkei":    "^N225",
            "USDCHF":    "USDCHF=X",
            "AUDUSD":    "AUDUSD=X",
            "Yield10Y":  "^TNX",
        },
        START_DATE, END_DATE
    )

    fred_pred = fetch_fred(
        {
            "FedFunds":      "FEDFUNDS",    # ставка ФРС
            "M2_USA":        "M2SL",        # денежная масса M2
            "UST_2Y":        "DGS2",        # 2Y Treasury
            "UST_10Y":       "DGS10",       # 10Y Treasury
            "Trade_Balance": "BOPGSTB",     # торговый баланс США
        },
        START_DATE, END_DATE
    )

    base = pd.concat([targets, market_pred], axis=1).sort_index()
    if not fred_pred.empty:
        fred_pred = resample_to_freq(fred_pred, "D")
        base = base.join(fred_pred, how="left").ffill()

    for col in ["EURUSD", "GBPUSD", "USDJPY"]:
        base = add_technical_features(base, col)

    common_pred = [c for c in base.columns
                   if c not in ["EURUSD", "GBPUSD", "USDJPY"]
                   and not any(c.startswith(p)
                               for p in ["EURUSD_", "GBPUSD_", "USDJPY_"])]

    for target in ["EURUSD", "GBPUSD", "USDJPY"]:
        if target not in base.columns:
            continue
        target_cols = [c for c in base.columns
                       if c == target or c.startswith(f"{target}_")]
        ds = base[target_cols + common_pred].dropna(subset=[target])
        save_dataset(ds, target, "B")


In [ ]:
build_group_b()

# ГРУППА C: МАКРО-ИНДИКАТОРЫ США

In [ ]:
def build_group_c():
    """
    Целевые ряды (месячные):
      - CPI (CPIAUCSL)
      - Industrial Production (INDPRO)
      - Потребительское доверие Michigan (UMCSENT)

    Предикторы:
      - PPI (PPIACO)                    — Producer Price Index
      - PCE (PCEPI)                     — Core PCE
      - Unemployment (UNRATE)
      - NFP (PAYEMS)                    — nonfarm payrolls
      - Fed Funds Rate (FEDFUNDS)
      - M2 Money Supply (M2SL)
      - Retail Sales (RSAFS)
      - Housing Starts (HOUST)
      - ISM PMI (NAPM)
      - Oil price (WTI monthly avg)
      - T10Y2Y — spread 10Y-2Y (инверсия кривой)
      - Технические фичи (MoM, YoY, лаги)
    """
    log.info("\n═══ ГРУППА C: Макро-индикаторы США ═══")

    fred_targets = fetch_fred(
        {
            "CPI":      "CPIAUCSL",
            "IndProd":  "INDPRO",
            "UMCSENT":  "UMCSENT",
        },
        START_DATE, END_DATE
    )

    fred_pred = fetch_fred(
        {
            "PPI":            "PPIACO",
            "PCE_core":       "PCEPILFE",
            "Unemployment":   "UNRATE",
            "NFP":            "PAYEMS",
            "FedFunds":       "FEDFUNDS",
            "M2":             "M2SL",
            "RetailSales":    "RSAFS",
            "HousingStarts":  "HOUST",
            "ISM_PMI":        "NAPM",
            "T10Y2Y":         "T10Y2Y",     # спред кривой
            "CredCardDebt":   "TOTALSL",    # потребкредит
            "Savings_Rate":   "PSAVERT",    # норма сбережений
        },
        START_DATE, END_DATE
    )

    # WTI месячный
    wti_daily = fetch_yf({"WTI_oil": "CL=F"}, START_DATE, END_DATE)
    if not wti_daily.empty:
        wti_monthly = wti_daily.resample("ME").mean()
        wti_monthly.index = wti_monthly.index.to_period("M").to_timestamp()
    else:
        wti_monthly = pd.DataFrame()

    # Объединяем на месячной сетке
    dfs = [fred_targets, fred_pred]
    if not wti_monthly.empty:
        dfs.append(wti_monthly)

    base = pd.concat(dfs, axis=1).sort_index()
    base.index = pd.to_datetime(base.index)
    base = base.resample("ME").last().ffill()

    # Добавляем MoM, YoY, лаги для целевых рядов
    for col in ["CPI", "IndProd", "UMCSENT"]:
        if col not in base.columns:
            continue
        s = base[col]
        base[f"{col}_MoM"]   = s.pct_change(1) * 100
        base[f"{col}_YoY"]   = s.pct_change(12) * 100
        for lag in [1, 2, 3, 6, 12]:
            base[f"{col}_lag{lag}"] = s.shift(lag)
        base[f"{col}_ma3"]  = s.rolling(3).mean()
        base[f"{col}_ma12"] = s.rolling(12).mean()
        base[f"{col}_std12"] = s.rolling(12).std()

    # Добавляем предикторы с лагами
    for col in ["PPI", "FedFunds", "M2", "Unemployment", "NFP"]:
        if col in base.columns:
            base[f"{col}_lag1"] = base[col].shift(1)
            base[f"{col}_lag3"] = base[col].shift(3)
            base[f"{col}_MoM"]  = base[col].pct_change(1) * 100
            base[f"{col}_YoY"]  = base[col].pct_change(12) * 100

    # Сохраняем 3 датасета
    all_pred = [c for c in base.columns
                if not any(c.startswith(t) for t in ["CPI", "IndProd", "UMCSENT"])]

    for target in ["CPI", "IndProd", "UMCSENT"]:
        if target not in base.columns:
            continue
        target_cols = [c for c in base.columns if c == target or c.startswith(f"{target}_")]
        ds = base[target_cols + all_pred].dropna(subset=[target])
        save_dataset(ds, target, "C")

In [ ]:
build_group_c()

# ГРУППА D: ЦИФРОВЫЕ АКТИВЫ

In [ ]:

def build_group_d():
    """
    Целевые ряды:
      - Bitcoin  (BTC-USD)
      - Ethereum (ETH-USD)

    Предикторы:
      - On-chain / market-structure прокси:
          BTC.Dominance (через BTC/ETH соотношение)
          Altcoin index proxy: SOL, BNB, XRP
      - Макро-рыночные:
          DXY, VIX, Gold, S&P500, NASDAQ
          Treasury 10Y
      - FRED:
          M2 (ликвидность)
          FedFunds
      - Технические фичи (часовая волатильность апроксимируется через daily range)
    """
    log.info("\n═══ ГРУППА D: Цифровые активы ═══")

    targets = fetch_yf(
        {"BTC": "BTC-USD", "ETH": "ETH-USD"},
        START_DATE, END_DATE
    )

    # Объёмы торгов (отдельная выгрузка)
    log.info("  yfinance: BTC/ETH объёмы торгов")
    btc_full = yf.download("BTC-USD", start=START_DATE, end=END_DATE,
                           progress=False, auto_adjust=True)
    eth_full = yf.download("ETH-USD", start=START_DATE, end=END_DATE,
                           progress=False, auto_adjust=True)

    # Обработка MultiIndex
    def extract_col(df, col):
        if isinstance(df.columns, pd.MultiIndex):
            return df[col].iloc[:, 0] if col in df.columns.get_level_values(0) else pd.Series(dtype=float)
        return df[col] if col in df.columns else pd.Series(dtype=float)

    vol_btc  = extract_col(btc_full, "Volume")
    high_btc = extract_col(btc_full, "High")
    low_btc  = extract_col(btc_full, "Low")
    vol_eth  = extract_col(eth_full, "Volume")
    high_eth = extract_col(eth_full, "High")
    low_eth  = extract_col(eth_full, "Low")

    crypto_extras = pd.DataFrame({
        "BTC_Volume":    vol_btc,
        "BTC_DailyRange": high_btc - low_btc,    # прокси внутридневной волатильности
        "ETH_Volume":    vol_eth,
        "ETH_DailyRange": high_eth - low_eth,
    })

    # Альткоины как предикторы
    alts = fetch_yf(
        {"SOL": "SOL-USD", "BNB": "BNB-USD", "XRP": "XRP-USD",
         "DOGE": "DOGE-USD"},
        START_DATE, END_DATE
    )

    market_pred = fetch_yf(
        {
            "DXY":     "DX-Y.NYB",
            "VIX":     "^VIX",
            "Gold":    "GC=F",
            "SP500":   "^GSPC",
            "NASDAQ":  "^IXIC",
            "Yield10Y":"^TNX",
        },
        START_DATE, END_DATE
    )

    fred_pred = fetch_fred(
        {
            "M2":       "M2SL",
            "FedFunds": "FEDFUNDS",
        },
        START_DATE, END_DATE
    )

    base = pd.concat([targets, crypto_extras, alts, market_pred], axis=1).sort_index()
    if not fred_pred.empty:
        fred_pred = resample_to_freq(fred_pred, "D")
        base = base.join(fred_pred, how="left").ffill()

    # BTC/ETH ratio
    if "BTC" in base.columns and "ETH" in base.columns:
        base["BTC_ETH_ratio"] = base["BTC"] / (base["ETH"] + 1e-9)

    # Технические фичи
    for col in ["BTC", "ETH"]:
        base = add_technical_features(base, col)

    # On-chain прокси: реализованная волатильность (21-дневная)
    for col in ["BTC", "ETH"]:
        if col in base.columns:
            ret = base[col].pct_change()
            base[f"{col}_RealVol21"] = ret.rolling(21).std() * np.sqrt(252)

    # Корреляция BTC-SP500 (60-дневная)
    if "BTC" in base.columns and "SP500" in base.columns:
        base["BTC_SP500_corr60"] = (
            base["BTC"].pct_change()
            .rolling(60)
            .corr(base["SP500"].pct_change())
        )

    common_pred = [c for c in base.columns
                   if not any(c.startswith(p) for p in ["BTC", "ETH"])]

    for target, prefix in [("BTC", "BTC_"), ("ETH", "ETH_")]:
        if target not in base.columns:
            continue
        target_cols = [c for c in base.columns
                       if c == target or c.startswith(prefix)]
        ds = base[target_cols + common_pred].dropna(subset=[target])
        save_dataset(ds, target, "D")


In [ ]:
build_group_d()